### Hugging Face Proxy + OpenAI Python SDK (Chat Completions)

This notebook demonstrates how to:
1. Create an `OpenAI` client that routes requests through the Hugging Face proxy.
2. Call a chat-completions model (e.g., `deepseek-ai/DeepSeek-R1`).
3. Remove any `<think>...</think>` blocks from the model output.


#### Prerequisites
- An API key available in the environment variable `AI_API_KEY`.
- Running this locally, set the powershell environment variable:
```powershell
$env:AI_API_KEY="sk-xxxx"
```

Step 1. Import required libraries

In [36]:
import os # For extracting API key from environment variables 
import re # For cleaning up the response from OpenAI
from openai import OpenAI # For making the connection to OpenAI

Step 2. Create client to OPENAI via the Hugging Face Proxy

In [37]:
# Create client to OpenAI via the Hugging Face Proxy
client = OpenAI(
    base_url="https://router.huggingface.co/v1",
    api_key=os.getenv("AI_API_KEY")
)

Step 3. Send a chat-completions request

In [38]:
completion = client.chat.completions.create(
    model="openai/gpt-oss-120b", # Available models at https://huggingface.co/models
    messages=[
        {
            "role": "system",
            "content": "Answer concisely. Provide an answer between 50-200 words."
        },
        {
            "role": "user",
            "content": "What is data engineering?"
        }
    ],
    temperature=0.3, # Control how consise the answer is using temperature
    max_tokens=120, # Control max number of tokens to use (could save cost)
)

raw_text = completion.choices[0].message.content
print(raw_text)

Data engineering is the discipline of designing, building, and maintaining the infrastructure that moves raw data from its source to a form usable for analysis, machine‑learning models, or business intelligence. Engineers create pipelines that ingest data (via APIs, logs, databases, streaming services), clean and transform it (validation, enrichment, aggregation), store it in scalable


Step 4. Remove the think blocks

In [39]:
# Use regex to remove the <think> section of the output
THINKING_PATTERN = re.compile(r"<think>.*?</think>", re.DOTALL)
def normalize_llm_output(text: str) -> str:
    """Strip any <think>...</think> blocks and trim whitespace."""
    text = THINKING_PATTERN.sub("", text)
    return text.strip()

# Apply the cleaning function to the output from step 3
cleaned_result = normalize_llm_output(raw_text)
print(cleaned_result)

Data engineering is the discipline of designing, building, and maintaining the infrastructure that moves raw data from its source to a form usable for analysis, machine‑learning models, or business intelligence. Engineers create pipelines that ingest data (via APIs, logs, databases, streaming services), clean and transform it (validation, enrichment, aggregation), store it in scalable
